In [22]:
import chromadb

In [23]:
client = chromadb.PersistentClient(path="./chroma_db")

In [24]:
collection = client.get_or_create_collection(
    name="laws_collection",
    metadata={"hnsw:space": "cosine"})

In [25]:
collection

Collection(name=laws_collection)

In [4]:
import json

text = []
metadata = []
ids = []

with open("ppc_laws.json", "r") as f:
    data = json.load(f)

    for id,section  in enumerate(data):
        
        if section["title"] is not None:
            
            ids.append(str(id))
            text.append(section["title"] + " " + section["body"])
            metadata.append({"title": section["title"]})
        else:
            ids.append(str(id))
            text.append(section["body"])
            metadata.append({"title": section["body"][:110]}) 

In [5]:
metadata[30:40]

[{'title': '34. Acts done by several persons In furtherance of common intention. When a criminal act is done by several pe'},
 {'title': '35. When such an act is criminal by reason of its being done with a criminal knowledge or intention'},
 {'title': '36. Effects caused partly by act and partly by omission'},
 {'title': '37. Co-operation by doing one of several acts constituting an offence'},
 {'title': '38. Persons concerned in criminal act may be guilty of different offences'},
 {'title': '39. "Voluntarily"'},
 {'title': '40. "Offence"'},
 {'title': '41. "Special law"'},
 {'title': '42. "Local Law"'},
 {'title': '43. "Illegal", "Legally bound to do"'}]

In [6]:
from sentence_transformers import SentenceTransformer

In [13]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', local_files_only=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3433.99it/s]


In [14]:
embeddings = embedding_model.encode(text).tolist()

In [16]:
len(embeddings)

537

In [26]:
collection.upsert(
    documents=text,
    metadatas=metadata,
    ids=ids,
    embeddings=embeddings
)

In [63]:
query = "What is punishment for theft under Pakistan Penal Code?"

query_embeddings = embedding_model.encode([query]).tolist()

In [65]:
results = collection.query(
    query_embeddings=query_embeddings,
    n_results=5,
    include=["metadatas", "documents"]
)

In [72]:
retreived_docs = results["documents"][0]

In [73]:
met = results["metadatas"][0]

In [27]:
### Tavily Web Search
import os
from langchain_tavily import TavilySearch
from dotenv import load_dotenv

load_dotenv()

os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

if __name__ == "__main__":
    query = "What is the punishment for theft in ppc?"
    results = tool.run(query)
    

In [31]:
results = tool.run(query)
results    

{'query': 'What is the punishment for theft in ppc?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.pasbanlawcollege.com/post/punishment-for-theft-under-ppc',
   'title': 'Punishment for Theft Under PPC - Pasban Law College',
   'content': 'The punishment for simple theft under the Pakistan Penal Code is generally imprisonment, which may extend to several years, or fine, or both',
   'score': 0.9183121,
   'raw_content': None},
  {'url': 'https://www.scribd.com/document/860698781/Assignment-on-Theft-and-Extortion-under-Pakistan-Penal-Code',
   'title': 'Theft vs. Extortion in Pakistan PPC | PDF - Scribd',
   'content': 'Theft is generally punishable under Section 379 with imprisonment up to three years, a fine, or both, but can extend up to 10 years for aggravated forms like',
   'score': 0.85097736,
   'raw_content': None},
  {'url': 'http://www.pljlawsite.com/html/ppc379.htm',
   'title': 'The Pakistan Penal Code, 1860 - PLJ Law Site'

In [29]:
# print(results['results'])
web_search_results = []
for result in results['results']:
    web_search_results.append({
        "title": result['title'],
        "content": result['content'],
        "url": result['url']
    })

In [30]:
web_search_results

[{'title': 'Punishment for Theft Under PPC - Pasban Law College',
  'content': 'The punishment for simple theft under the Pakistan Penal Code is generally imprisonment, which may extend to several years, or fine, or both',
  'url': 'https://www.pasbanlawcollege.com/post/punishment-for-theft-under-ppc'},
 {'title': 'Theft vs. Extortion in Pakistan PPC | PDF - Scribd',
  'content': 'Theft is generally punishable under Section 379 with imprisonment up to three years, a fine, or both, but can extend up to 10 years for aggravated forms like',
  'url': 'https://www.scribd.com/document/860698781/Assignment-on-Theft-and-Extortion-under-Pakistan-Penal-Code'},
 {'title': 'The Pakistan Penal Code, 1860 - PLJ Law Site',
  'content': 'Whoever commits theft shall be punished with imprisonment of either description for a term which may extend to three years, or with fine, or with both. [1]. The',
  'url': 'http://www.pljlawsite.com/html/ppc379.htm'},
 {'title': 'Sections 378-379 - United Nations Offi